# Heat Model Training & Scoring (Python Notebook)

This notebook trains a simple linear regression heat-score model from historical engagement data and scores posts in realtime.

It expects a CSV with columns: `text`, `likes`, `reposts`, `replies`.


## Heat prediction model (historical training + realtime scoring)

This section adds a lightweight, **standard-library only** heat-score model. It trains from a historical dataset (CSV) and can score incoming posts in real time.

**Expected CSV columns:** `text`, `likes`, `reposts`, `replies`.

The model uses a simple ridge-regularized linear regression over text features (length, word count, hashtags, mentions, URLs, punctuation).


In [ ]:
import csv
import re
from dataclasses import dataclass
from pathlib import Path

@dataclass
class HeatModel:
    feature_names: list[str]
    weights: list[float]
    bias: float
    mean: list[float]
    std: list[float]

FEATURE_NAMES = [
    'text_length',
    'word_count',
    'hashtag_count',
    'mention_count',
    'url_count',
    'exclamation_count',
    'question_count',
]

def extract_heat_features(text: str) -> list[float]:
    trimmed = text.strip()
    words = trimmed.split() if trimmed else []
    hashtags = re.findall(r'#[\w]+', trimmed)
    mentions = re.findall(r'@[\w.]+', trimmed)
    urls = re.findall(r'https?://\S+', trimmed)
    exclamations = re.findall(r'!', trimmed)
    questions = re.findall(r'\?', trimmed)
    return [
        float(len(trimmed)),
        float(len(words)),
        float(len(hashtags)),
        float(len(mentions)),
        float(len(urls)),
        float(len(exclamations)),
        float(len(questions)),
    ]

def parse_heat_csv(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open('r', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            rows.append({
                'text': row.get('text', '') or '',
                'likes': float(row.get('likes', 0) or 0),
                'reposts': float(row.get('reposts', 0) or 0),
                'replies': float(row.get('replies', 0) or 0),
            })
    return rows

def engagement_score(row: dict) -> float:
    return max(0.0, row['likes'] + row['reposts'] * 2.0 + row['replies'] * 1.5)

def compute_mean_std(features: list[list[float]]) -> tuple[list[float], list[float]]:
    count = len(features[0]) if features else 0
    mean = [0.0] * count
    std = [0.0] * count
    for row in features:
        for idx, value in enumerate(row):
            mean[idx] += value
    mean = [value / len(features) for value in mean]
    for row in features:
        for idx, value in enumerate(row):
            std[idx] += (value - mean[idx]) ** 2
    std = [(value / len(features)) ** 0.5 or 1.0 for value in std]
    return mean, std

def standardize(features: list[list[float]], mean: list[float], std: list[float]) -> list[list[float]]:
    return [
        [(value - mean[idx]) / std[idx] for idx, value in enumerate(row)]
        for row in features
    ]

def transpose(matrix: list[list[float]]) -> list[list[float]]:
    return [list(col) for col in zip(*matrix)]

def matmul(a: list[list[float]], b: list[list[float]]) -> list[list[float]]:
    return [
        [sum(a[i][k] * b[k][j] for k in range(len(b))) for j in range(len(b[0]))]
        for i in range(len(a))
    ]

def matvec(matrix: list[list[float]], vector: list[float]) -> list[float]:
    return [sum(value * vector[idx] for idx, value in enumerate(row)) for row in matrix]

def invert(matrix: list[list[float]]) -> list[list[float]]:
    size = len(matrix)
    augmented = [row[:] + [1.0 if i == j else 0.0 for j in range(size)] for i, row in enumerate(matrix)]
    for i in range(size):
        pivot = augmented[i][i]
        if pivot == 0:
            swap = next((idx for idx in range(i + 1, size) if augmented[idx][i] != 0), None)
            if swap is None:
                raise ValueError('Matrix is not invertible')
            augmented[i], augmented[swap] = augmented[swap], augmented[i]
            pivot = augmented[i][i]
        for j in range(len(augmented[i])):
            augmented[i][j] /= pivot
        for k in range(size):
            if k == i:
                continue
            factor = augmented[k][i]
            for j in range(len(augmented[k])):
                augmented[k][j] -= factor * augmented[i][j]
    return [row[size:] for row in augmented]

def train_heat_model(rows: list[dict], ridge: float = 1e-6) -> HeatModel:
    if not rows:
        raise ValueError('No training data provided')
    features = [extract_heat_features(row['text']) for row in rows]
    targets = [engagement_score(row) for row in rows]
    mean, std = compute_mean_std(features)
    standardized = standardize(features, mean, std)
    design = [row + [1.0] for row in standardized]
    design_t = transpose(design)
    xtx = matmul(design_t, design)
    for idx in range(len(xtx) - 1):
        xtx[idx][idx] += ridge
    xty = matvec(design_t, targets)
    inverse = invert(xtx)
    weights = matvec(inverse, xty)
    return HeatModel(
        feature_names=FEATURE_NAMES,
        weights=weights[:-1],
        bias=weights[-1],
        mean=mean,
        std=std,
    )

def predict_heat(model: HeatModel, text: str) -> float:
    features = extract_heat_features(text)
    standardized = [(value - model.mean[idx]) / model.std[idx] for idx, value in enumerate(features)]
    score = sum(value * model.weights[idx] for idx, value in enumerate(standardized)) + model.bias
    return max(0.0, score)


In [ ]:
dataset_path = Path('assets/datasets/heat-training.csv')
if dataset_path.exists():
    training_rows = parse_heat_csv(dataset_path)
else:
    training_rows = [
        {'text': 'Bluesky launch day! #bluesky', 'likes': 120, 'reposts': 40, 'replies': 12},
        {'text': 'Check out this new feature update: https://example.com', 'likes': 45, 'reposts': 12, 'replies': 6},
        {'text': 'What do you think about the new UI?', 'likes': 30, 'reposts': 5, 'replies': 14},
        {'text': 'Wow!!! This is incredible!!!', 'likes': 80, 'reposts': 18, 'replies': 9},
        {'text': 'Join our community meetup tonight @bsky', 'likes': 25, 'reposts': 4, 'replies': 3},
    ]

heat_model = train_heat_model(training_rows)
heat_model


In [ ]:
scored_posts = [
    {
        'text': post['text'],
        'created_at': post['created_at'],
        'score': predict_heat(heat_model, post['text']),
    }
    for post in sample_posts
]

sorted(scored_posts, key=lambda item: item['score'], reverse=True)[:5]
